# Customer Data Analysis

This notebook demonstrates connecting to PostgreSQL, querying customer data, and creating visualizations using Matplotlib.

In [ ]:
# Import required libraries
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Connect to PostgreSQL Database

In [ ]:
# Database connection parameters
conn_params = {
    'host': 'localhost',
    'port': 5432,
    'database': 'customerdb',
    'user': 'postgres',
    'password': 'postgres'
}

# Establish connection
try:
    conn = psycopg2.connect(**conn_params)
    print("✅ Successfully connected to PostgreSQL database!")
except Exception as e:
    print(f"❌ Error connecting to database: {e}")

## 2. Query Customer Data

In [ ]:
# Query all customers
query_customers = "SELECT * FROM customers ORDER BY signup_date;"
df_customers = pd.read_sql_query(query_customers, conn)

print(f"Total customers: {len(df_customers)}")
df_customers.head()

In [ ]:
# Query all orders
query_orders = "SELECT * FROM orders ORDER BY order_date;"
df_orders = pd.read_sql_query(query_orders, conn)

print(f"Total orders: {len(df_orders)}")
df_orders.head()

## 3. Customer Distribution by Region

In [ ]:
# Count customers by region
region_counts = df_customers['region'].value_counts()

# Create bar chart
plt.figure(figsize=(10, 6))
region_counts.plot(kind='bar', color='steelblue')
plt.title('Customer Distribution by Region', fontsize=16, fontweight='bold')
plt.xlabel('Region', fontsize=12)
plt.ylabel('Number of Customers', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nCustomer counts by region:")
print(region_counts)

## 4. Customer Segments Analysis

In [ ]:
# Count customers by segment
segment_counts = df_customers['customer_segment'].value_counts()

# Create pie chart
plt.figure(figsize=(8, 8))
colors = ['#ff9999', '#66b3ff', '#99ff99']
plt.pie(segment_counts.values, labels=segment_counts.index, autopct='%1.1f%%', 
        colors=colors, startangle=90)
plt.title('Customer Segments Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCustomer counts by segment:")
print(segment_counts)

## 5. Order Trends Over Time

In [ ]:
# Convert order_date to datetime
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])

# Group by month and count orders
df_orders['year_month'] = df_orders['order_date'].dt.to_period('M')
monthly_orders = df_orders.groupby('year_month').size()

# Create line chart
plt.figure(figsize=(12, 6))
monthly_orders.plot(kind='line', marker='o', color='green', linewidth=2, markersize=8)
plt.title('Order Trends Over Time', fontsize=16, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Number of Orders', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Revenue Analysis by Product Category

In [ ]:
# Calculate revenue by product category
category_revenue = df_orders.groupby('product_category')['order_amount'].sum().sort_values(ascending=False)

# Create horizontal bar chart
plt.figure(figsize=(10, 6))
category_revenue.plot(kind='barh', color='coral')
plt.title('Total Revenue by Product Category', fontsize=16, fontweight='bold')
plt.xlabel('Revenue ($)', fontsize=12)
plt.ylabel('Product Category', fontsize=12)
plt.tight_layout()
plt.show()

print("\nRevenue by category:")
print(category_revenue)

## 7. Customer Lifetime Value by Segment

In [ ]:
# Join customers and orders to calculate CLV
query_clv = """
SELECT 
    c.customer_segment,
    COUNT(DISTINCT c.customer_id) as customer_count,
    COUNT(o.order_id) as total_orders,
    SUM(o.order_amount) as total_revenue,
    AVG(o.order_amount) as avg_order_value
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_segment
ORDER BY total_revenue DESC;
"""

df_clv = pd.read_sql_query(query_clv, conn)

# Display the results
print("Customer Lifetime Value by Segment:")
print(df_clv)

# Create grouped bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Total revenue by segment
df_clv.plot(x='customer_segment', y='total_revenue', kind='bar', ax=ax1, color='purple', legend=False)
ax1.set_title('Total Revenue by Customer Segment', fontsize=14, fontweight='bold')
ax1.set_xlabel('Customer Segment', fontsize=12)
ax1.set_ylabel('Total Revenue ($)', fontsize=12)
ax1.tick_params(axis='x', rotation=45)

# Average order value by segment
df_clv.plot(x='customer_segment', y='avg_order_value', kind='bar', ax=ax2, color='teal', legend=False)
ax2.set_title('Average Order Value by Customer Segment', fontsize=14, fontweight='bold')
ax2.set_xlabel('Customer Segment', fontsize=12)
ax2.set_ylabel('Avg Order Value ($)', fontsize=12)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 8. Close Database Connection

In [ ]:
# Close the connection
conn.close()
print("✅ Database connection closed.")